In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [6]:
%pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
df = pd.read_csv("../datasets/tutorials.csv")


In [8]:
df.head()

,Video_Title,Video_Description,Upload_Date,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,Actual_Quality_Score
0,Learn Python in 10 Minutes!,Super fast Python tutorial for absolute beginn...,2023-01-15,10,980000,12000,800,150000,28
1,Complete Python Bootcamp 2024,Comprehensive Python course covering all funda...,2023-02-10,480,520000,41000,3200,210000,88
2,Machine Learning Crash Course,Everything about ML in 5 minutes. Covers neura...,2023-01-20,5,1200000,9500,600,300000,18
3,Deep Learning with PyTorch,Step by step PyTorch tutorial with real projec...,2023-03-05,360,280000,24000,1800,180000,85
4,Web Dev in 1 Hour,Full stack web development explained in just 6...,2023-01-25,60,870000,11000,750,220000,32


In [9]:
df.shape

(138, 9)

In [10]:
df['like_to_view_ratio'] = df['Like_Count'] / df['View_Count']

In [11]:
df['comment_to_view_ratio'] = df['Comment_Count'] / df['View_Count']

In [12]:
df['engagement_rate'] = (df['Like_Count'] + df['Comment_Count']) / df['View_Count']

In [13]:
df['title_length'] = df['Video_Title'].apply(len)

In [14]:
df.head()

,Video_Title,Video_Description,Upload_Date,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,Actual_Quality_Score,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length
0,Learn Python in 10 Minutes!,Super fast Python tutorial for absolute beginn...,2023-01-15,10,980000,12000,800,150000,28,0.012245,0.000816,0.013061,27
1,Complete Python Bootcamp 2024,Comprehensive Python course covering all funda...,2023-02-10,480,520000,41000,3200,210000,88,0.078846,0.006154,0.085000,29
2,Machine Learning Crash Course,Everything about ML in 5 minutes. Covers neura...,2023-01-20,5,1200000,9500,600,300000,18,0.007917,0.000500,0.008417,29
3,Deep Learning with PyTorch,Step by step PyTorch tutorial with real projec...,2023-03-05,360,280000,24000,1800,180000,85,0.085714,0.006429,0.092143,26
4,Web Dev in 1 Hour,Full stack web development explained in just 6...,2023-01-25,60,870000,11000,750,220000,32,0.012644,0.000862,0.013506,17


In [15]:
clickbait_words = ['minute', 'overnight', 'instantly', 'secret', 'trick', 'rich', 'cheat', 'replace', 'blow']

In [16]:
df['is_clickbait'] = df['Video_Title'].str.lower().apply(
    lambda title: int(any(word in title for word in clickbait_words))
)

features = the columns we give to the model to learn from

target = the column we want the model to predict (Actual_Quality_Score)

In [17]:
features = [
    'Duration_Minutes',
    'View_Count',
    'Like_Count',
    'Comment_Count',
    'Channel_Subscriber_Count',
    'like_to_view_ratio',
    'comment_to_view_ratio',
    'engagement_rate',
    'title_length',
    'is_clickbait'
]

xfeatures = df[features]
y = df['Actual_Quality_Score']

In [18]:
xfeatures.head()

,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,like_to_view_ratio,comment_to_view_ratio,engagement_rate,title_length,is_clickbait
0,10,980000,12000,800,150000,0.012245,0.000816,0.013061,27,1
1,480,520000,41000,3200,210000,0.078846,0.006154,0.085000,29,0
2,5,1200000,9500,600,300000,0.007917,0.000500,0.008417,29,0
3,360,280000,24000,1800,180000,0.085714,0.006429,0.092143,26,0
4,60,870000,11000,750,220000,0.012644,0.000862,0.013506,17,0



I split the data into:
- Training set (80%) — the model learns from this
- Test set (20%) — we check how well the model does on unseen data

In [19]:
X_train, X_test, y_train, y_test = train_test_split(xfeatures, y, test_size=0.2, random_state=42)

In [20]:
X_train.shape, X_test.shape

((110, 10), (28, 10))

Train Model 1: Linear Regression

In [21]:
model = LinearRegression()

In [22]:
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [23]:
ypred = model.predict(X_test)

In [24]:
r2  = round(r2_score(y_test, ypred), 3)


In [25]:
r2

0.966

In [26]:
mae = round(mean_absolute_error(y_test,ypred), 2)

In [27]:
mae

4.06

Train Model 2: Random Forest

In [28]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [29]:
rf_preds = rf_model.predict(X_test)


In [30]:
rf_r2  = round(r2_score(y_test, rf_preds), 3)
rf_mae = round(mean_absolute_error(y_test, rf_preds), 2)

In [31]:
comparison = pd.DataFrame({
    'Model' : ['Linear Regression', 'Random Forest'],
    'R2'    : [r2, rf_r2],
    'MAE'   : [mae, rf_mae]
})

comparison

,Model,R2,MAE
0,Linear Regression,0.966,4.06
1,Random Forest,0.974,3.50


R² tells us what fraction of the variation in quality the model explains.

R² = 1.0 would be perfect. R² close to 0 means the model is no better than just guessing the average.

MAE (Mean Absolute Error) tells us on average how many points off our predictions are.

##### Feature Importance

Random Forest tells us which features it relied on the most.

This answers: **what makes a tutorial clickbait vs genuinely good?**

In [32]:
importance = pd.Series(rf_model.feature_importances_, index=features)

In [33]:
importance = importance.sort_values(ascending=False).round(4)

In [34]:
importance

comment_to_view_ratio       0.2538
engagement_rate             0.2303
like_to_view_ratio          0.2285
View_Count                  0.1893
Duration_Minutes            0.0498
Channel_Subscriber_Count    0.0408
title_length                0.0028
Like_Count                  0.0023
Comment_Count               0.0023
is_clickbait                0.0001
dtype: float64

The most important feature for detecting a bad tutorial.

In [35]:
importance.idxmax()

'comment_to_view_ratio'

In [36]:
df['predicted_score'] = rf_model.predict(xfeatures)


In [37]:
df[['Video_Title', 'Actual_Quality_Score', 'predicted_score']].head(10)

,Video_Title,Actual_Quality_Score,predicted_score
0,Learn Python in 10 Minutes!,28,26.79
1,Complete Python Bootcamp 2024,88,80.80
2,Machine Learning Crash Course,18,17.73
3,Deep Learning with PyTorch,85,85.41
4,Web Dev in 1 Hour,32,30.01
5,React JS Full Course,87,86.66
6,JavaScript for Beginners,82,82.02
7,Become a Data Scientist Overnight!,15,14.04
8,SQL Complete Tutorial,83,81.71
9,C++ in 20 Minutes,12,14.65


In [38]:
top10 = df.nlargest(10, 'predicted_score')[['Video_Title', 'predicted_score', 'Actual_Quality_Score', 'Duration_Minutes', 'engagement_rate']]

In [39]:
top10

,Video_Title,predicted_score,Actual_Quality_Score,Duration_Minutes,engagement_rate
114,Intro to Compilers,91.47,92,480,0.116875
133,Distributed Systems,91.31,92,480,0.115345
124,Probabilistic Graphical Models,90.81,91,440,0.123333
98,Statistical Learning Theory,90.70,91,420,0.113636
137,Information Retrieval Systems,90.45,90,420,0.119778
119,Mathematical Optimization,88.69,89,380,0.111552
25,Computer Networks Complete Course,88.38,90,420,0.103182
19,Operating Systems Fundamentals,88.30,91,480,0.108421
11,Algorithms and Data Structures,88.06,89,540,0.093182
74,Linear Algebra for Data Science,87.67,90,420,0.104091


In [40]:
top10 = top10.reset_index(drop=True)


In [41]:
top10.index = top10.index + 1
top10

,Video_Title,predicted_score,Actual_Quality_Score,Duration_Minutes,engagement_rate
1,Intro to Compilers,91.47,92,480,0.116875
2,Distributed Systems,91.31,92,480,0.115345
3,Probabilistic Graphical Models,90.81,91,440,0.123333
4,Statistical Learning Theory,90.70,91,420,0.113636
5,Information Retrieval Systems,90.45,90,420,0.119778
6,Mathematical Optimization,88.69,89,380,0.111552
7,Computer Networks Complete Course,88.38,90,420,0.103182
8,Operating Systems Fundamentals,88.30,91,480,0.108421
9,Algorithms and Data Structures,88.06,89,540,0.093182
10,Linear Algebra for Data Science,87.67,90,420,0.104091


In [42]:
bottom10 = df.nsmallest(10, 'predicted_score')[['Video_Title', 'predicted_score', 'Actual_Quality_Score', 'View_Count', 'like_to_view_ratio']]
bottom10 = bottom10.reset_index(drop=True)
bottom10.index = bottom10.index + 1
bottom10

,Video_Title,predicted_score,Actual_Quality_Score,View_Count,like_to_view_ratio
1,Interview Cheat Sheet One Page,6.99,5,2200000,0.007273
2,ChatGPT Will Replace Programmers,8.48,7,2300000,0.007391
3,Solve Leetcode While You Sleep,8.69,6,1550000,0.006452
4,Coding Will Make You Rich!,9.22,8,2100000,0.007143
5,Earn Money with AI Today,9.62,8,2000000,0.007000
6,Become 10x Programmer Instantly,9.96,10,2000000,0.007250
7,I Made $100K Freelancing,10.06,9,1800000,0.007222
8,Make $10000 with Coding,10.30,9,1900000,0.006842
9,Crack Any Coding Interview Tomorrow,10.60,10,1800000,0.006667
10,This One Line of Code Will Blow Your Mind,11.70,11,2100000,0.007619


- Worst tutorials have very **high view counts** but very **low like-to-view ratios**
- This is exactly the clickbait pattern — people click, watch for 30 seconds, then leave without liking

In [43]:
output = df[['Video_Title', 'Upload_Date', 'Duration_Minutes', 'View_Count',
             'Like_Count', 'Comment_Count', 'Channel_Subscriber_Count',
             'engagement_rate', 'like_to_view_ratio', 'is_clickbait',
             'Actual_Quality_Score', 'predicted_score']].copy()

In [44]:
output['Rank'] = output['predicted_score'].rank(ascending=False).astype(int)


In [45]:
output = output.sort_values('Rank')
output = output.reset_index(drop=True)

In [46]:
output.to_csv("tutorial_scores.csv", index=False)

In [47]:
output.head(15)

,Video_Title,Upload_Date,Duration_Minutes,View_Count,Like_Count,Comment_Count,Channel_Subscriber_Count,engagement_rate,like_to_view_ratio,is_clickbait,Actual_Quality_Score,predicted_score,Rank
0,Intro to Compilers,2023-06-26,480,48000,5200,410,36000,0.116875,0.108333,0,92,91.47,1
1,Distributed Systems,2023-06-30,480,58000,6200,490,43000,0.115345,0.106897,0,92,91.31,2
2,Probabilistic Graphical Models,2023-07-22,440,42000,4800,380,31000,0.123333,0.114286,0,91,90.81,3
3,Statistical Learning Theory,2023-07-12,420,55000,5800,450,41000,0.113636,0.105455,0,91,90.70,4
4,Information Retrieval Systems,2023-07-26,420,45000,5000,390,33000,0.119778,0.111111,0,90,90.45,5
5,Mathematical Optimization,2023-07-28,380,58000,6000,470,43000,0.111552,0.103448,0,89,88.69,6
6,Computer Networks Complete Course,2023-06-20,420,110000,10500,850,82000,0.103182,0.095455,0,90,88.38,7
7,Operating Systems Fundamentals,2023-06-01,480,95000,9500,800,75000,0.108421,0.100000,0,91,88.30,8
8,Algorithms and Data Structures,2023-03-20,540,220000,19000,1500,130000,0.093182,0.086364,0,89,88.06,9
9,Linear Algebra for Data Science,2023-06-25,420,88000,8500,660,65000,0.104091,0.096591,0,90,87.67,10
